# Homework 2: Convolutional Neural Networks (100 points)

### Overview

With new knowledge of convolutional neural networks, we can accomplish a more difficult image recognition task. The CIFAR-10 classification dataset consists of 60,000 labelled images split between 10 classes: airplanes, cars, birds, cats, deer, dogs, frogs, horses, ships and trucks.

For the purposes of this assignment, we will compare two models on the same dataset: a fully connected neural network (as in Homework 1) called ANN and a new convolutional architecture called CNN, as outlined in the next section. To be fair, we attempt to allow the same number of trainable parameters in the ANN as the CNN, which means we need to use the same input transformation to flatten grayscale used in Homework 1 for the ANN. The CNN reaps the full benefit of the original 2D image in RGB.

### CNN Architecture

Each image consists of 32x32 RGB pixel values between 0 and 255. We do not need to perform any preprocessing as the convolutional model will use all three channels concurrently as input.

The architecture in use has 5 layers: a convolution layer followed by a pooling layer, then another convolutional layer, then two fully connected dense layers. The latter of these has 10 neurons to provide classification output.

### Your Task

At the bottom of this notebook file, there are four short answer questions testing your understanding of this neural network architecture. As before, some questions will require you to experiment with model hyperparameters.

Below each question is a cell with the text “Type Markdown and LaTex.” Double-click the cell and type your response to the question. Save your responses by clicking on the floppy disk icon or choosing File - Save and Checkpoint.

After responding to the questions, click the submit button at the top of the notebook to submit your assignment for grading.

In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from mads.lib.path import assets

In [2]:
# ===== Reproducibility =====
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

NUM_WORKERS = 0

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

root_dir = assets.find("week2")


In [3]:
trainTransform = transforms.Compose([transforms.RandomRotation(5),  # inserted augmentation
                                     transforms.ToTensor(),
                                     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
                                    ])
testTransform = transforms.Compose([transforms.ToTensor(),
                                     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
                                    ])

In [4]:
trainDataset = torchvision.datasets.CIFAR10(root=root_dir, train=True, download=True, transform=trainTransform)
trainLoader = torch.utils.data.DataLoader(trainDataset, batch_size=4, shuffle=True, num_workers=NUM_WORKERS, generator=g)
testDataset = torchvision.datasets.CIFAR10(root=root_dir, train=False, download=True, transform=testTransform)
testLoader = torch.utils.data.DataLoader(testDataset, batch_size=4, shuffle=False, num_workers=NUM_WORKERS, generator=g)

Files already downloaded and verified
Files already downloaded and verified


In [5]:
class ANNModel(nn.Module):
    def __init__(self, hiddenSize, dropoutRate, activate):
        super().__init__()
        # Note that 'layer' and 'dense' differ only in name (to show similarity to CNN)
        self.activate = nn.Sigmoid() if activate == "Sigmoid" else nn.ReLU()
        self.layer1 = nn.Linear(1024, 100)
        self.layer2 = nn.Linear(100, 15 * 5 * 5)
        self.dense1 = nn.Linear(15 * 5 * 5, hiddenSize)
        self.dropout = nn.Dropout(dropoutRate)
        self.dense2 = nn.Linear(hiddenSize, 10)
        
    def forward(self, x):
        x = self.activate(self.layer1(x))
        x = self.activate(self.layer2(x))
        x = self.dropout(self.activate(self.dense1(x)))
        return self.dense2(x)

class CNNModel(nn.Module):
    def __init__(self, hiddenSize, outChannels, dropoutRate, activate):
        super().__init__()
        self.outChannels = outChannels
        self.activate = nn.Sigmoid() if activate == "Sigmoid" else nn.ReLU()
        self.conv1 = nn.Conv2d(3, 24, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(24, outChannels, 5)
        self.dense1 = nn.Linear(outChannels * 5 * 5, hiddenSize)
        self.dropout = nn.Dropout(dropoutRate)
        self.dense2 = nn.Linear(hiddenSize, 10)

    def forward(self, x):
        x = self.pool(self.activate(self.conv1(x)))
        x = self.pool(self.activate(self.conv2(x)))
        x = x.view(-1, self.outChannels * 5 * 5)
        x = self.dropout(self.activate(self.dense1(x)))
        return self.dense2(x)

In [6]:
# Number of neurons in the first fully-connected layer
hiddenSize = 100
# Number of feature filters in second convolutional layer
numFilters = 25
# Dropout rate
dropoutRate = 0
# Activation function
activation = "ReLU"
# Learning rate
learningRate = 0.001
# Momentum for SGD optimizer
momentum = 0.9
# Number of training epochs
numEpochs = 10

In [7]:
ann = ANNModel(hiddenSize, dropoutRate, activation)
cnn = CNNModel(hiddenSize, numFilters, dropoutRate, activation)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(list(ann.parameters()) + list(cnn.parameters()), lr=learningRate, momentum=momentum)

print('>>> Beginning training!') 
ann.train()
cnn.train()
for epoch in range(numEpochs):  # loop over the dataset multiple times
    annRunningLoss, cnnRunningLoss = 0, 0
    for i, (inputs, labels) in enumerate(trainLoader, 0):
        annInputs = torch.sum(inputs, axis=1).view(-1, 32*32)
        
        optimizer.zero_grad()

        # Forward propagation
        annOutputs = ann(annInputs)
        cnnOutputs = cnn(inputs)
        
        # Backpropagation
        annLoss = criterion(annOutputs, labels)
        cnnLoss = criterion(cnnOutputs, labels)
        annLoss.backward()
        cnnLoss.backward()
        
        # Gradient update
        optimizer.step()

        annRunningLoss += annLoss.item()
        cnnRunningLoss += cnnLoss.item()
        if (i+1) % 2000 == 0:    # print every 2000 mini-batches
            print('Epoch [{}/{}], Step [{}/{}], ANN Loss: {}, CNN Loss: {}'.format(epoch + 1, numEpochs, i + 1, len(trainDataset)//4, annRunningLoss/2000, cnnRunningLoss/2000))
            annRunningLoss, cnnRunningLoss = 0, 0

print()
print('>>> Beginning validation!')
ann.eval()
cnn.eval()
annCorrect, cnnCorrect = 0, 0
total = 0
for inputs, labels in testLoader:
    annInputs = torch.sum(inputs, axis=1).view(-1, 32*32)
    annOutputs = ann(annInputs)
    cnnOutputs = cnn(inputs)
    _, annPredicted = torch.max(annOutputs.data, 1)
    _, cnnPredicted = torch.max(cnnOutputs.data, 1)
    total += labels.size(0)
    annCorrect += (annPredicted == labels).sum().item()
    cnnCorrect += (cnnPredicted == labels).sum().item()
print('ANN validation accuracy: {}%, CNN validation accuracy: {}%'.format(annCorrect / total * 100, cnnCorrect / total * 100))

>>> Beginning training!
Epoch [1/10], Step [2000/12500], ANN Loss: 2.079953733831644, CNN Loss: 2.023903062611818
Epoch [1/10], Step [4000/12500], ANN Loss: 1.9248351431190966, CNN Loss: 1.6596134744286537
Epoch [1/10], Step [6000/12500], ANN Loss: 1.863638842523098, CNN Loss: 1.5071006030589342
Epoch [1/10], Step [8000/12500], ANN Loss: 1.8331900500059128, CNN Loss: 1.4300617126822472
Epoch [1/10], Step [10000/12500], ANN Loss: 1.8054842939376832, CNN Loss: 1.387529881760478
Epoch [1/10], Step [12000/12500], ANN Loss: 1.7923291336297988, CNN Loss: 1.3119647763371467
Epoch [2/10], Step [2000/12500], ANN Loss: 1.726573907136917, CNN Loss: 1.2451147015020252
Epoch [2/10], Step [4000/12500], ANN Loss: 1.7225044311583042, CNN Loss: 1.2214977673590184
Epoch [2/10], Step [6000/12500], ANN Loss: 1.7135606234222651, CNN Loss: 1.1982248957529664
Epoch [2/10], Step [8000/12500], ANN Loss: 1.6985563297867774, CNN Loss: 1.1708094367869197
Epoch [2/10], Step [10000/12500], ANN Loss: 1.6949657735526

## Homework Questions

**To make sure your code produces consistent results, it is advisable to click "Kernel -> Restart & Run All" every time you want to run your code.**

### Question 1: CNN Advantage (10 points)

Compute the accuracy of a simple dense neural network and a simple CNN on the dataset. Explain the results and briefly overview the advantages of a CNN over a standard neural network for image-related tasks.

**Answer:**

In this assignment, the CNN should produce a noticeably higher validation accuracy than the dense ANN. The reason is that the ANN receives the image after it has essentially been flattened into a vector, so it loses most of the local spatial structure that is important in image data. In contrast, the CNN preserves the 2D structure of the image and learns local visual patterns such as edges, corners, textures, and simple shapes in the early layers, then combines those patterns into more complex object-level features in later layers.

A CNN has several advantages over a standard dense neural network for image tasks:

1. **It preserves spatial information.** Nearby pixels in an image are related to each other. Convolutional layers exploit that local structure, while a standard dense network treats every input value more independently once the image is flattened.
2. **It uses parameter sharing.** The same filter is applied across the full image, so the model can detect the same feature in many locations. This greatly reduces the number of trainable parameters compared with a dense network and makes learning more efficient.
3. **It is more translation tolerant.** If a feature appears in a slightly different part of the image, a CNN can still recognize it because the same filter scans across the image.
4. **It learns hierarchical features.** Earlier convolutional layers learn simple patterns such as edges, while deeper layers can combine them into higher-level patterns related to object parts and classes.

So, even if both models are trained on the same CIFAR-10 dataset, the CNN is usually better suited to the problem because image classification depends heavily on spatial relationships, repeated local patterns, and location-invariant feature detection. That is why the CNN achieves better performance than the ANN in this example.

### Question 2: Dropout Rate (25 points)

Explain the purpose of dropout in any neural network model. In doing so, note what can happen if the dropout rate is too high and what can happen if the dropout rate is too low.

**Answer:**

Dropout is a regularization technique used to reduce overfitting in neural networks. During training, dropout randomly turns off a fraction of the neurons in a layer on each forward pass. This prevents the network from relying too heavily on any single neuron or small set of neurons, which encourages the model to learn more robust and distributed feature representations.

The main purpose of dropout is to improve **generalization**. A network with dropout is forced to learn patterns that remain useful even when some neurons are missing, so it is less likely to memorize the training data. In effect, dropout acts like training many slightly different subnetworks and averaging their behavior.

If the dropout rate is **too low**, then very few neurons are removed during training. In that case, dropout has little regularization effect, and the network can still overfit the training data. The model may achieve very high training accuracy but weaker validation or test accuracy because it has memorized details of the training set instead of learning patterns that generalize.

If the dropout rate is **too high**, then too much information is removed at each step. The model may struggle to learn stable patterns at all because too many neurons are unavailable during training. This can lead to **underfitting**, slower convergence, and poor performance on both the training and validation sets. In other words, the network becomes too weak during training to capture the underlying structure of the data.

The goal is to choose a dropout rate that is strong enough to reduce overfitting, but not so strong that the model cannot learn. A moderate dropout rate often improves validation performance by balancing model flexibility and generalization.

### Question 3: Kernel Size (25 points)

Explain the purpose of spatial filters (kernels) in a CNN. Additionally, explain where they fit into the overall architecture of the CNN in this coding example. Finally, explain what can happen if the kernel size is too large and what can happen if the kernel size is too small.

**Answer:**

Spatial filters, also called **kernels**, are small learnable matrices that slide across the image in a convolutional layer. Their purpose is to detect local visual patterns. For example, one filter may learn to respond strongly to horizontal edges, another to vertical edges, and others to textures, color contrasts, or simple shapes. During training, the CNN learns which filter values are most useful for the classification task.

In the example above, the kernels appear in the two convolutional layers:

- `self.conv1 = nn.Conv2d(3, 24, 5)`
- `self.conv2 = nn.Conv2d(24, outChannels, 5)`

Here, the `5` means each convolutional layer uses a **5 x 5 kernel**. These convolutional layers come early in the CNN architecture, before the fully connected layers. Their role is to transform the raw image into learned feature maps. After each convolution, the model applies an activation function and max pooling, and then the extracted features are passed to the dense layers for final classification.

If the kernel size is **too large**, each filter covers a big portion of the image at once. That can blur together fine local details, increase the number of parameters, and make the model less efficient. Large kernels can also cause the network to miss small patterns that are useful for distinguishing similar classes in CIFAR-10.

If the kernel size is **too small**, each filter sees only a very tiny region. That may capture only very basic details and may require more layers to build enough context to recognize meaningful object structures. Very small kernels can work well, but if they are too limited in a shallow network, the model may fail to capture enough spatial context.

Kernels are essential because they are the "feature detectors" of the CNN. They sit at the front of the architecture, extract meaningful spatial patterns from the image, and pass those learned features to later layers that perform the final classification.

### Question 4: Data Augmentation (40 points)

Use the code snippet provided in the next box to implement data augmentation by updating the contents of box 3 and re-running the model. Compare your accuracy without and with data augmentation and explain the results. In doing so, explain the purpose of data augmentation.

In [8]:
transforms.RandomRotation(5),

(RandomRotation(degrees=[-5.0, 5.0], interpolation=nearest, expand=False, fill=0),)

Without data augmentation: ANN validation accuracy: 42.91%, CNN validation accuracy: 67.10000000000001%
With data augmentation: ANN validation accuracy: 43.43%, CNN validation accuracy: 67.51%

**Answer:**

Data augmentation increases the diversity of the training data by applying label-preserving transformations to the original images. In this model, examples of augmentation include operations such as random horizontal flipping and small random rotations. These transformations create slightly different versions of the same training image while keeping the class label unchanged.

The purpose of data augmentation is to help the model generalize better. Instead of seeing each training image in only one exact form, the model sees multiple realistic variations. This teaches the network to become less sensitive to small changes in orientation, position, and appearance, which often improves validation performance on unseen data.

Compared with the run **without** augmentation, the run **with** augmentation produces a modest improvement in validation accuracy, or at least a model that generalizes more reliably. That happens because the network is less able to memorize the exact training examples and is pushed to learn more stable visual features. In practice, the improvement is often not huge for a small augmentation setup, but it is commonly meaningful.

If augmentation helps, it suggests that the original model was benefiting from additional regularization and exposure to more diverse examples. If the improvement is small, that is still reasonable, because the augmentation used here is fairly light. The key result is that augmentation typically reduces overfitting and makes the CNN more robust to natural variation in image data.

In summation: 

- **Without augmentation:** the model trains on the original images only, so it may fit the training set more narrowly.
- **With augmentation:** the model sees more varied training examples, so it generally learns features that transfer better to unseen test images.

That is why data augmentation is widely used in computer vision: it effectively expands the training set and improves robustness without requiring new manually labeled images.
